# CLONING REPOSITORIES

In [ ]:
!git clone https://github.com/Alessandro-Carotenuto/synt_satellite_vqgan_from_TT
%cd synt_satellite_vqgan_from_TT
!python setup.py

In [ ]:
from kaggle_secrets import UserSecretsClient
secret_label = "WANDB_API_KEY"
secret_value = UserSecretsClient().get_secret(secret_label)

# CONFIGURATION

In [ ]:
print("Config overwrite")
import config
# Training
config.DATA_ROOT = "/kaggle/input/datasets/carotenutoalessandro/cvusa-groundandpolar-subset-35191"
config.NUM_EPOCHS = 2
config.LEARNING_RATE = 5e-4
config.BATCH_SIZE = 8
config.LEARNING_RATE_MODE = config.LRMODE.REDUCEONPLATEAU

#ARCHITECTURE OPTIONS
config.DROPOUT=0.2
config.TOKEN_MASKING_SCHEDULING_START=1.0
config.TOKEN_MASKING_SCHEDULING_END=1.0
config.USE_ROPE = True

#BACKBONE
config.BACKBONE_SIZE = config.BBSIZE.SMALL  # LARGE = 16384 codebook, SMALL = 1024 codebook

#LOGGING
config.USE_WANDB = True
config.RUN_NAME = "61->62 ROPE RedLRplat 1024"
config.WANDB_GROUP = 'ROPE RedLRplat 1024'
config.IDENTIFIER = 'ACEngineering'

print("Config overwrite completed")

loading=True

# TRAIN-TRANSFORMER MAIN TRAINING LOOP

In [ ]:
if loading==False:
    import train_transformer
    print("Main Training:")
    train_transformer.main()

# RESUME

In [ ]:
if loading==True:
    import config
    import train_transformer
    from taming_interface import load_with_optimizer
    from CVUSA_Manager import CVUSADataset

    # 1. Checkpoint da cui riprendere
    RESUME_CHECKPOINT = "/kaggle/input/models/acengineering/60-epochs-rope-1024/pytorch/default/1/CVUSAGround2Satellite_improved_epoch60_loss4.409_20260621_093311.pth"

    # 2. Carica checkpoint con optimizer e scheduler ripristinati
    model, optimizer, scheduler, checkpoint_info, device = load_with_optimizer(
        checkpoint_path=RESUME_CHECKPOINT,
        kaggle_flag=config.KAGGLE_FLAG,
        lr=config.LEARNING_RATE
    )
    print(f"✅ Ripreso da epoch {checkpoint_info['epoch']}, loss {checkpoint_info['loss']:.4f}")

    # 3. Dataloaders
    train_loader, test_loader = CVUSADataset.create_dataloaders(
        data_root=config.DATA_ROOT,
        batch_size=config.BATCH_SIZE
    )

    start_epoch = checkpoint_info['epoch']

    train_transformer.train_model_with_evaluation(
      model, train_loader, test_loader,
      num_epochs=config.NUM_EPOCHS,
      lr=config.LEARNING_RATE,
      optimizer=optimizer,
      scheduler=scheduler,
      start_epoch=start_epoch
    )

# LOAD TO INFERENCE

In [ ]:
import os
from inference import test_inference
from taming_interface import load_saved_model

#LOAD THE BEST MODEL
directory = "."
matches = [ file for file in os.listdir(directory) if "improve" in os.path.splitext(file)[0].lower()]

best_model_name=matches[0]
model, checkpoint_info, device = load_saved_model(checkpoint_path=best_model_name, kaggle_flag=config.KAGGLE_FLAG)
model.eval()
print(f"Inference Test using weights from Epoch {checkpoint_info['epoch']}")
test_inference(model, data_root=config.DATA_ROOT, device=device)